# Diagonal A/E/T component inference

## Aim

Fit the A, E, and T auto-PSDs jointly over $10^{-4}$--$10^{-1}\,\mathrm{Hz}$, using a separate free noise spline for each channel and shared Galactic parameters. No cross-spectrum is estimated or inverted.

$$S_c(t,f)=S_{{\rm noise},c}(t,f)+A_{\rm gal}T_{{\rm gal},c}(t,f;f_{\rm knee}),\qquad c\in\{A,E,T\}.$$

All valid bins enter the likelihood. The broad few-mHz valleys are noise--Galaxy crossovers and remain in all diagnostics. Only simulation-identified neighborhoods around genuine narrow transfer minima may be omitted from broad whitening summaries; they remain fitted and remain in surface-recovery metrics.


## 1. Model

The orthonormal transformation is

$$A=(Z-X)/\sqrt2,\quad E=(X-2Y+Z)/\sqrt6,\quad T=(X+Y+Z)/\sqrt3.$$

For each channel,

$$\log S_{{\rm noise},c}(t,f)=\log\mu_c+B_t(t)W_cB_f(f)^\mathsf T.$$

$\mu_c$ is one scalar weak-calibration level, derived here from the geometric median of the OMS/TM prediction in that channel. The matrices $W_c$ are independent P-spline coefficients and must learn the full time-frequency noise shapes. The analytic OMS/TM surface is not passed to the likelihood.

The Galactic amplitude and knee frequency are shared across A/E/T. The response template is treated as known. Spline roughness precisions are fixed in this study, while the unpenalized spline modes receive broad proper priors centred on $\mu_c$.

The likelihood is the product of three scalar Whittle likelihoods. This diagonal approximation does not estimate the A/E/T cross-spectral density and does not assert statistical independence for realistic data.


## 2. Run the full-band posterior

```bash
cd /Users/avi/Documents/projects/wdm_psd/lisa_data_generation
../wdm_psd/.venv/bin/python run_aet_diagonal_pilot.py \
  --fmin-hz 1e-4 --fmax-hz 1e-1 --component-fmax-hz 1e-1 \
  --warmup 300 --samples 300 --frequency-bin-size 64 \
  --phi-time 100 --phi-frequency 100 --noise-level-log-sd 5 \
  --max-tree-depth 10 \
  --require-convergence
```

The cells below read the saved full-band artifact; they do not repeat NUTS. A short warmup/sample run is only an execution smoke test and must not be reported as a posterior result.


In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / "combined_esa_xyz.h5").exists():
    ROOT = ROOT / "lisa_data_generation"
ARTIFACT = ROOT / "component_models_aet_diagonal_fullband_fitall_weakcal_nuts.npz"
SURFACE_PLOT = ROOT / "pspline_aet_plots" / "aet_diagonal_fullband_fitall_component_recovery.png"
PARAMETER_PLOT = ROOT / "pspline_aet_plots" / "aet_diagonal_fullband_fitall_parameter_posterior.png"

if ARTIFACT.exists():
    result = np.load(ARTIFACT, allow_pickle=True)
    print(f"loaded: {ARTIFACT}")
else:
    result = None
    print("Full-band production artifact is not present yet; run the command above.")


## 3. Posterior summary

The next cell reports the fitted frequency range, sampler diagnostics, shared Galactic parameters, and channel-by-channel recovery metrics directly from the artifact. Component errors are evaluated only where that component contributes at least 3% of the injected total. Pointwise interval fractions from one realization are descriptive rather than calibrated coverage probabilities.

Do not copy numerical values from the earlier 20 mHz calibrated-surface pilot: it used a different noise model and a narrower band.

### Current execution status

A 20-warmup/20-draw two-chain full-band smoke run completed and generated both posterior-plot paths, confirming the data and model plumbing. It was deliberately non-converged (maximum $\hat R=11.46$, minimum ESS $=1.03$, minimum E-BFMI $=0.12$) and is not a scientific result. An initial 300/300 attempt with maximum tree depth 12 was stopped after more than 30 minutes before producing an artifact. The documented command now caps tree depth at 10 so saturation is measured rather than allowing unbounded validation runtime. A converged production artifact is still pending.


In [ ]:
if result is not None:
    diagnostics = dict(
        zip(result["diagnostic_names"].tolist(), result["diagnostic_values"].tolist())
    )
    metrics = dict(zip(result["metric_names"].tolist(), result["metric_values"].tolist()))
    amplitude = result["amplitude_draws"]
    knee_mhz = 1e3 * result["f_knee_hz_draws"]
    summary = {
        "fitted band [Hz]": [
            float(result["fitted_frequency_min_hz"]),
            float(result["fitted_frequency_max_hz"]),
        ],
        "posterior status": str(result["posterior_status"]),
        "posterior usable": bool(result["posterior_usable"]),
        "noise prior levels": result["noise_prior_level_psd"].tolist(),
        "A_gal median": float(np.median(amplitude)),
        "A_gal 90%": np.quantile(amplitude, [0.05, 0.95]).tolist(),
        "injected A_gal": float(result["injected_amplitude"]),
        "f_knee median [mHz]": float(np.median(knee_mhz)),
        "f_knee 90% [mHz]": np.quantile(knee_mhz, [0.05, 0.95]).tolist(),
        "injected f_knee [mHz]": 1e3 * float(result["injected_f_knee_hz"]),
        **diagnostics,
        **metrics,
    }
    print(json.dumps(summary, indent=2))
else:
    print("Posterior summary pending a converged full-band production run.")


## 4. Posterior figures

**Figure 1.** Time-median total and component auto-PSDs for A, E, and T. The total panel compares the posterior median and pointwise 90% interval with the injection. The component panel compares the free noise spline and parametric Galactic foreground with simulation truth. The dotted line marks the noise--Galaxy crossover; shaded diagnostic regions, if present, are omitted only from broad whitening summaries.

**Figure 2.** Joint and marginal posterior for the shared Galactic amplitude and knee frequency. Their covariance is part of the scientific result and should be shown together with the injected values in this controlled study.


In [ ]:
for path in (SURFACE_PLOT, PARAMETER_PLOT):
    if path.exists():
        figure = plt.figure(figsize=(12, 8 if path == SURFACE_PLOT else 4))
        plt.imshow(plt.imread(path))
        plt.axis("off")
        plt.show()
        plt.close(figure)
    else:
        print(f"pending figure: {path}")


## 5. Interpretation and manuscript notes

### What this analysis tests

- Whether three independent free A/E/T noise splines can coexist with one shared parametric Galactic foreground.
- Whether weak channel-level OMS/TM scale information is sufficient without supplying the analytic noise shape.
- Whether the diagonal model recovers the total and component auto-PSDs in the controlled archive over $10^{-4}$--$10^{-1}\,\mathrm{Hz}$.

### Diagnostic boundary

All valid bins are fitted. The broad few-mHz crossover valleys remain in whitening and recovery diagnostics. Only genuine narrow response-minimum neighborhoods may be omitted from continuum mean-$z^2$ summaries, and that mask is derived from simulation information. The fit itself must still confront those bins.

### What remains conditional

The Galactic response and sky map are assumed known. The OMS/TM calculation supplies only three scalar prior centres, not a surface. The archive's XYZ Galactic channels were generated independently and then rotated under a zero-XYZ-cross-spectrum contract, so this is a controlled diagonal-auto-PSD experiment rather than a realistic multichannel stochastic-response validation. Robustness to prior width, spline complexity, response mismatch, correlated A/E/T data, and random seed remains to be tested.

### Draft analysis paragraph

We transformed the simulated XYZ data to orthonormal A, E, and T channels and modelled only their diagonal WDM auto-powers over $10^{-4}$--$10^{-1}\,\mathrm{Hz}$. Each channel received an independent tensor-product log-P-spline instrumental-noise surface, while a response-informed Galactic foreground shared amplitude and knee-frequency parameters across channels. Analytic OMS/TM predictions were reduced to one geometric-mean PSD level per channel to centre broad proper priors; their time-frequency shapes did not enter the likelihood. The joint posterior was sampled with two-chain NUTS. Genuine response-minimum neighborhoods were retained in the likelihood and component-recovery errors and omitted, when flagged, only from broad continuum-whitening summaries.
